In [ ]:
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.size":        11,
    "axes.titlesize":   11,
    "axes.labelsize":   11,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "legend.fontsize":  9,
})

In [ ]:
results_A, results_B = {}, {}

for path in sorted(glob.glob('../results/*_7kl_*_history_A.json')):
    name = os.path.basename(path).replace('_history_A.json', '')
    with open(path) as f:
        results_A[name] = json.load(f)

for path in sorted(glob.glob('../results/*_7kl_*_history_B.json')):
    name = os.path.basename(path).replace('_history_B.json', '')
    with open(path) as f:
        results_B[name] = json.load(f)

print(f"A: {len(results_A)} modeli, B: {len(results_B)} modeli")

def parse_name(name):
    parts      = name.split("_")
    resnet     = parts[0][1:]
    n_unfreeze = int(parts[2][1:])
    n_linear   = int(parts[3][1:])
    return resnet, n_unfreeze, n_linear

epochs = np.arange(1, 21)

color_resnet   = {"18": "tab:blue",   "34": "tab:red"}
color_unfreeze = {0: "#6692EA",       1: "#A68898",     2: "#E57E46"}
color_linear   = {1: "tab:cyan",      2: "tab:olive",   3: "tab:pink"}
color_pressure = {"A": "tab:purple",  "B": "tab:green"}

row_labels = [
    "ResNet size",
    "N unfreeze",
    "N linear",
    "Class pressure (A=sampler, B=criterion)",
]

fig, axes = plt.subplots(4, 2, figsize=(20, 20), dpi=300, sharey=False)

def draw_row(ax_train, ax_test, color_fn):
    for name, history in results_A.items():
        color = color_fn(name, "A")
        train_er = 1 - np.array(history["train_acc"])
        test_er  = 1 - np.array(history["test_acc"])
        ax_train.plot(epochs, train_er, color=color, lw=1.0, alpha=0.65)
        ax_test.plot( epochs, test_er,  color=color, lw=1.0, alpha=0.65)

    for name, history in results_B.items():
        color = color_fn(name, "B")
        train_er = 1 - np.array(history["train_acc"])
        test_er  = 1 - np.array(history["test_acc"])
        ax_train.plot(epochs, train_er, color=color, lw=1.0, alpha=0.65)
        ax_test.plot( epochs, test_er,  color=color, lw=1.0, alpha=0.65)

draw_row(axes[0,0], axes[0,1],
         lambda name, ab: color_resnet[parse_name(name)[0]])
draw_row(axes[1,0], axes[1,1],
         lambda name, ab: color_unfreeze[parse_name(name)[1]])
draw_row(axes[2,0], axes[2,1],
         lambda name, ab: color_linear[parse_name(name)[2]])
draw_row(axes[3,0], axes[3,1],
         lambda name, ab: color_pressure[ab])

legends = [
    [Line2D([0],[0], color="tab:blue", lw=2, label="ResNet18"),
     Line2D([0],[0], color="tab:red",  lw=2, label="ResNet34")],

    [Line2D([0],[0], color="#6692EA", lw=2, label="Unfreezed layers: 0"),
     Line2D([0],[0], color="#A68898", lw=2, label="Unfreezed layers: 1"),
     Line2D([0],[0], color="#E57E46", lw=2, label="Unfreezed layers: 2")],

    [Line2D([0],[0], color="tab:cyan",  lw=2, label="Linear layers: 1"),
     Line2D([0],[0], color="tab:olive", lw=2, label="Linear layers: 2"),
     Line2D([0],[0], color="tab:pink",  lw=2, label="Linear layers: 3")],

    [Line2D([0],[0], color="tab:purple", lw=2, label="Class pressure on sampler"),
     Line2D([0],[0], color="tab:green",  lw=2, label="Class pressure on criterion")],
]

for i, (row_label, leg) in enumerate(zip(row_labels, legends)):
    for j, col_title in enumerate(["Train Error Rate (7 classes)", "Test Error Rate (7 classes)"]):
        ax = axes[i, j]
        ax.set_yscale("log")
        ax.set_yticks([1.00, 0.50, 0.30, 0.20, 0.10, 0.05, 0.02])
        ax.set_ylim(0.02, 1.00)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x*100:.0f}%"))
        ax.xaxis.set_major_locator(plt.MultipleLocator(5))
        ax.yaxis.set_minor_locator(plt.MultipleLocator(0.05))
        ax.xaxis.set_minor_locator(plt.MultipleLocator(1))
        ax.grid(True, which="major", linestyle="-", linewidth=0.5, alpha=0.5)
        ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.3)
        ax.set_xlabel("Epoch")
        if j == 0:
            ax.set_ylabel(row_label)
        if i == 0:
            ax.set_title(col_title)
    axes[i, 1].legend(handles=leg, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
results_bin_A, results_bin_B = {}, {}

for path in sorted(glob.glob('../results/*_bin_*_history_A.json')):
    name = os.path.basename(path).replace('_history_A.json', '')
    with open(path) as f:
        results_bin_A[name] = json.load(f)

for path in sorted(glob.glob('../results/*_bin_*_history_B.json')):
    name = os.path.basename(path).replace('_history_B.json', '')
    with open(path) as f:
        results_bin_B[name] = json.load(f)

fig, axes = plt.subplots(4, 4, figsize=(20, 20), dpi=300)

col_titles  = ["Melanoma Sensitivity (binary model)", "Melanoma Specificity (binary model)", "Youden's J (binary model)", "AUC-ROC (binary model)"]
row_labels  = ["ResNet size", "N unfreeze", "N linear", "Class pressure"]

def get_metrics(history):
    sensitivity = [TP/(TP+FN) if (TP+FN) > 0 else 0
                   for TP, FN in zip(history["TP"], history["FN"])]
    specificity = [TN/(TN+FP) if (TN+FP) > 0 else 0
                   for TN, FP in zip(history["TN"], history["FP"])]
    J           = [s + sp - 1 for s, sp in zip(sensitivity, specificity)]
    auc         = history["auc"]
    return sensitivity, specificity, J, auc

def draw_row_4(axes_row, color_fn):
    for name, history in results_bin_A.items():
        color = color_fn(name, "A")
        s, sp, J, auc = get_metrics(history)
        axes_row[0].plot(epochs, s,   color=color, lw=1.0, alpha=0.65)
        axes_row[1].plot(epochs, sp,  color=color, lw=1.0, alpha=0.65)
        axes_row[2].plot(epochs, J,   color=color, lw=1.0, alpha=0.65)
        axes_row[3].plot(epochs, auc, color=color, lw=1.0, alpha=0.65)

    for name, history in results_bin_B.items():
        color = color_fn(name, "B")
        s, sp, J, auc = get_metrics(history)
        axes_row[0].plot(epochs, s,   color=color, lw=1.0, alpha=0.65)
        axes_row[1].plot(epochs, sp,  color=color, lw=1.0, alpha=0.65)
        axes_row[2].plot(epochs, J,   color=color, lw=1.0, alpha=0.65)
        axes_row[3].plot(epochs, auc, color=color, lw=1.0, alpha=0.65)

draw_row_4(axes[0], lambda name, ab: color_resnet[parse_name(name)[0]])
draw_row_4(axes[1], lambda name, ab: color_unfreeze[parse_name(name)[1]])
draw_row_4(axes[2], lambda name, ab: color_linear[parse_name(name)[2]])
draw_row_4(axes[3], lambda name, ab: color_pressure[ab])

for i, row_label in enumerate(row_labels):
    for j, col_title in enumerate(col_titles):
        ax = axes[i, j]
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x*100:.0f}%"))
        ax.xaxis.set_major_locator(plt.MultipleLocator(5))
        ax.yaxis.set_minor_locator(plt.MultipleLocator(0.05))
        ax.xaxis.set_minor_locator(plt.MultipleLocator(1))
        ax.grid(True, which="major", linestyle="-", linewidth=0.5, alpha=0.5)
        ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.5)
        ax.set_xlabel("Epoch")
        if j == 0:
            ax.set_ylabel(row_label)
        if i == 0:
            ax.set_title(col_title)

axes[0, 3].legend(handles=[
    Line2D([0],[0], color="tab:blue", lw=2, label="ResNet18"),
    Line2D([0],[0], color="tab:red",  lw=2, label="ResNet34"),
])

axes[1, 3].legend(handles=[
    Line2D([0],[0], color="#6692EA", lw=2, label="Unfreezed layers: 0"),
    Line2D([0],[0], color="#A68898", lw=2, label="Unfreezed layers: 1"),
    Line2D([0],[0], color="#E57E46", lw=2, label="Unfreezed layers: 2"),
])

axes[2, 3].legend(handles=[
    Line2D([0],[0], color="tab:cyan",  lw=2, label="Linear layers: 1"),
    Line2D([0],[0], color="tab:olive", lw=2, label="Linear layers: 2"),
    Line2D([0],[0], color="tab:pink",  lw=2, label="Linear layers: 3"),
])

axes[3, 3].legend(handles=[
    Line2D([0],[0], color="tab:purple", lw=2, label="Class pressure on sampler"),
    Line2D([0],[0], color="tab:green",  lw=2, label="Class pressure on criterion"),
])

plt.tight_layout()
plt.show()